In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from itertools import chain

In [0]:
bronze_path = "abfss://bronze@logisticdatalakestorage.dfs.core.windows.net/restcountries/"
silver_path = "abfss://silver@logisticdatalakestorage.dfs.core.windows.net/restcountries/"

###  Read Bronze

In [0]:
df = spark.read.json(bronze_path)
df.limit(2).display()
print(f'Bronze rows: {df.count()}')
df.printSchema()

capital,currencies,name,region,timezones,year,month,day
List(Singapore),"List(null, null, null, null, null, null, List(Singapore dollar, $), null, null)","List(Singapore, List(null, null, List(Singapore, Republic of Singapore), null, null, List(Singapura, Republik Singapura), null, List(சிங்கப்பூர், சிங்கப்பூர் குடியரசு), null, List(新加坡, 新加坡共和国)), Republic of Singapore)",Asia,List(UTC+08:00),2026,5,27
List(New Delhi),"List(null, null, null, null, null, List(Indian rupee, ₹), null, null, null)","List(India, List(null, null, List(India, Republic of India), null, List(भारत, भारत गणराज्य), null, null, List(இந்தியா, இந்தியக் குடியரசு), null, null), Republic of India)",Asia,List(UTC+05:30),2026,5,27


Bronze rows: 10
root
 |-- capital: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- currencies: struct (nullable = true)
 |    |-- AED: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol: string (nullable = true)
 |    |-- AUD: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol: string (nullable = true)
 |    |-- BRL: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol: string (nullable = true)
 |    |-- EUR: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol: string (nullable = true)
 |    |-- GBP: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol: string (nullable = true)
 |    |-- INR: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol: string (nullable = true)
 |    |-- SGD: struct (nullable = true)
 |    |

### Flatten + Add country_code

In [0]:
country_name_to_iso2 = {
    'Singapore':            'SG',
    'India':                'IN',
    'United Kingdom':       'GB',
    'United States':        'US',
    'Thailand':             'TH',
    'United Arab Emirates': 'AE',
    'Australia':            'AU',
    'France':               'FR',
    'Brazil':               'BR',
    'Germany':              'DE'
}

In [0]:
mapping_expr = create_map([lit(x) for x in chain(*country_name_to_iso2.items())])

In [0]:
df_silver = df.select(
    col('name.common').alias('country_name'),
    col('name.official').alias('official_name'),
    col('region'),
    col('timezones')[0].alias('timezone'),
    col('capital')[0].alias('capital'),
    col('year'), col('month'), col('day')
)

In [0]:
df_silver = df_silver.withColumn('country_code', mapping_expr.getItem(col('country_name'))) \
    .filter(col('country_code').isNotNull()) \
    .dropDuplicates(['country_code']) \
    .withColumn('ingestion_date', current_date())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/column.py:527: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


In [0]:
print(f'Silver rows: {df_silver.count()}')
df_silver.display()

Silver rows: 10


country_name,official_name,region,timezone,capital,year,month,day,country_code,ingestion_date
United Arab Emirates,United Arab Emirates,Asia,UTC+04:00,Abu Dhabi,2026,5,27,AE,2026-05-27
Australia,Commonwealth of Australia,Oceania,UTC+05:00,Canberra,2026,5,27,AU,2026-05-27
Brazil,Federative Republic of Brazil,Americas,UTC-05:00,Brasília,2026,5,27,BR,2026-05-27
Germany,Federal Republic of Germany,Europe,UTC+01:00,Berlin,2026,5,27,DE,2026-05-27
France,French Republic,Europe,UTC-10:00,Paris,2026,5,27,FR,2026-05-27
United Kingdom,United Kingdom of Great Britain and Northern Ireland,Europe,UTC-08:00,London,2026,5,27,GB,2026-05-27
India,Republic of India,Asia,UTC+05:30,New Delhi,2026,5,27,IN,2026-05-27
Singapore,Republic of Singapore,Asia,UTC+08:00,Singapore,2026,5,27,SG,2026-05-27
Thailand,Kingdom of Thailand,Asia,UTC+07:00,Bangkok,2026,5,27,TH,2026-05-27
United States,United States of America,Americas,UTC-12:00,"Washington, D.C.",2026,5,27,US,2026-05-27


### Write to Silver Delta Lake

In [0]:
df_silver.write.format('delta').mode('overwrite').partitionBy('year', 'month', 'day').save(silver_path)